In [1]:
#!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source:": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source:': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

c:\Users\lakba\anaconda3\envs\ai-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python Basics for RAG\n\nPython is the most widely used language for building RAG systems because of its strong ecosystem in AI, data processing, and web development.\n\n1. Variables and Data Types\nname = "RAG System"\nversion = 1.0\nis_active = True\n\nCommon data types:\n\nint, float\nstr\nbool\nlist, dict, tuple\n2. Control Flow\nif version > 0:\n    print("Valid version")\nelse:\n    print("Invalid version")\n\nLoops:\n\nfor i in range(3):\n    print(i)\n3. Functions\ndef greet(user):\n    return f"Hello, {user}"\n\nprint(greet("Developer"))\n4. Working with Lists and Dictionaries\ndocuments = ["doc1", "doc2", "doc3"]\n\nfor doc in documents:\n    print(doc)\n\nmetadata = {\n    "source": "pdf",\n    "pages": 10\n}\n5. File Handling (Important for RAG)\nwith open("data.txt", "r") as file:\n    content = file.read()\n    print(content)\n6. Installing Libraries\npip install numpy pandas langchain openai faiss-cpu\n7. Ke

In [9]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

# Ingestion Pipeline

In [11]:
# Data -> Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## Documents

In [12]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("Total PDFs:", num_docs)
    print("Total Pages:", len(all_docs))
    return all_docs

In [13]:
all_pdf_documents = load_all_pdfs()

Total PDFs: 3
Total Pages: 40


In [14]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

## Chunks

In [15]:
# Chunks
# !pip install langchain_text_splitters

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [17]:
chunks = split_docs(all_pdf_documents)

In [18]:
len(chunks)

468

## Embedding

In [19]:
from sentence_transformers import SentenceTransformer

In [20]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [21]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3645.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


C:\Users\lakba\AppData\Local\Temp\ipykernel_31672\4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## Vector Store

In [22]:
import chromadb
import uuid

In [23]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [24]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 1608


In [25]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches: 100%|██████████| 15/15 [00:29<00:00,  1.99s/it]


embeddings shape: (468, 384)
total documents added in vector store= 468
docs in collection: 2076


# Retrieval Pipeline

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("What is Machine Learning")

Batches: 100%|██████████| 1/1 [00:00<00:00, 30.64it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_9ab034c0-d461-48a8-aeff-88e165f6a664',
  'document': 'Figure 1. General terminology used in this paper  For instance, in the field of statistics the focus is on statistical learning, which is defined as a set of me-thods and algorithms to gain knowledge, predict outcomes, and make decisions by constructing models from a data set [12]. From a statistics point of view, machine learning can be regarded as an implemen-tation of statistical learning [13].  Within the field of computer science, machine learning has the focus of designing efficient',
  'metadata': {'keywords': '',
   'doc_index': 127,
   'creator': 'Word',
   'source': 'data/pdfs\\Research2.pdf',
   'page': 1,
   'aapl:keywords': '[]',
   'content_length': 493,
   'moddate': "D:20180921202506Z00'00'",
   'title': 'Microsoft Word - HICSS2019_Kuehl_Goutier_Hirt_Satzger_AI_final.docx',
   'producer': 'Mac OS X 10.13.6 Quartz PDFContext',
   'total_pages': 10,
   'creationdate': "D:20180921202506Z00'00'",
   'page_la

# Integrate with LLMs

## Google Gimini

In [ ]:
API_KEY_GIMINI = " YOUR API KEY HERE"

In [31]:
# !pip install langchain-google-genai google-generativeai

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

os.environ["GOOGLE_API_KEY"] = API_KEY_GIMINI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_output_tokens=1024
)

In [36]:
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)
    
    context = "\n".join(doc["document"] for doc in results) if results else ""
    
    prompt = f"""
You are an AI assistant.

Use the given context to answer the question in a detailed and explanatory way.

- Explain in 4-5 lines
- Use simple language
- Add examples if possible

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)
    return response.content

In [37]:
answer = generate_output("What is AI?", rag_retriever, llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 60.11it/s]


embeddings shape: (1, 384)
retrieved 3 documents
Artificial Intelligence (AI) refers to machines and robots that possess human-like intelligence. This means they have the ability to reason, understand information, find patterns, and learn from past experiences.

For example, an AI system can learn to recognize faces in photos or recommend products based on your past purchases. This advanced capability of machines is creating significant changes and new opportunities in areas like business and government.
